In [14]:
import tensorflow as tf
import numpy as np
import os
import cv2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [15]:
os.environ["PYTHONHASHSEED"] = str(42)
tf.random.set_seed(42)
np.random.seed(42)

In [16]:
IMAGE_SIZE = 1536
PATCH_SIZE = 256
BATCH_SIZE = 2
EPOCHS = 4
LEARNING_RATE = 0.0001

In [17]:
noisy_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\noisy_data\DIV2K\DIV2K_train_HR\DIV2K_train_HR"
clean_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\clean_data\DIV2K\DIV2K_train_HR\DIV2K_train_HR"

clean_test_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\clean_data\DIV2K\DIV2K_valid_HR\DIV2K_valid_HR"
noisy_test_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\noisy_data\DIV2K\DIV2K_valid_HR\DIV2K_valid_HR"



# noisy_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\noisy_data\DIV2K\DIV2K_train_HR"
# clean_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\clean_data\DIV2K\DIV2K_train_HR"

# clean_test_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\clean_data\DIV2K\DIV2K_valid_HR"
# noisy_test_path = r"C:\Users\91909\Desktop\ML\DATA\NTIRE\noisy_data\DIV2K\DIV2K_valid_HR"

In [18]:
def load_image_generator(path):
    for file in os.listdir(path):
        img = cv2.imread(os.path.join(path, file))
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        img = img.astype(np.float32) / 255.0
        yield img

In [19]:
def extract_patches(img):
    patches = []
    for i in range(0, IMAGE_SIZE, PATCH_SIZE):
        for j in range(0, IMAGE_SIZE, PATCH_SIZE):
            patch = img[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            if patch.shape[:2] == (PATCH_SIZE, PATCH_SIZE):
                patches.append(patch)
    return np.array(patches)

In [20]:
def data_generator(noisy_path, clean_path):
    for noisy_img, clean_img in zip(load_image_generator(noisy_path), load_image_generator(clean_path)):
        noisy_patches = extract_patches(noisy_img)
        clean_patches = extract_patches(clean_img)
        for i in range(len(noisy_patches)):
            yield noisy_patches[i], clean_patches[i]

In [21]:
dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(noisy_path, clean_path),
    output_signature=(tf.TensorSpec(shape=(PATCH_SIZE, PATCH_SIZE, 3), dtype=tf.float32),
                      tf.TensorSpec(shape=(PATCH_SIZE, PATCH_SIZE, 3), dtype=tf.float32))
)

In [22]:
dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.experimental.AUTOTUNE)

In [23]:
inp = Input(shape=(PATCH_SIZE, PATCH_SIZE, 3))
x = Conv2D(64, (3, 3), padding='same')(inp)
x = Activation('relu')(x)
for _ in range(11):
    x = Conv2D(64, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
x = Conv2D(3, (3, 3), padding='same')(x)
output = tf.keras.layers.Subtract()([inp, x])

model = Model(inputs=inp, outputs=output)

In [24]:
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d_13 (Conv2D)             (None, 256, 256, 64  1792        ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 activation_12 (Activation)     (None, 256, 256, 64  0           ['conv2d_13[0][0]']              
                                )                                                           

In [25]:
model.compile(optimizer=Adam(LEARNING_RATE), loss='mse')
early_stopping = EarlyStopping(monitor='loss', patience=3, restore_best_weights=True)

In [26]:
model.fit(dataset, epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stopping])

Epoch 1/4


14400/14400 [==============================] - 2681s 186ms/step - loss: 0.0044
Epoch 2/4
14400/14400 [==============================] - 2850s 198ms/step - loss: 0.0020
Epoch 3/4
14400/14400 [==============================] - 2613s 181ms/step - loss: 0.0018
Epoch 4/4
14400/14400 [==============================] - 2634s 183ms/step - loss: 0.0017


In [35]:
model.save("tf_denoiser_2.h5")

In [28]:
def denoise_and_stitch(noisy_test_path):
    denoised_images = []
    for img in load_image_generator(noisy_test_path):
        patches = extract_patches(img)
        denoised_patches = model.predict(patches)
        stitched_image = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3))
        count = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 1))
        idx = 0
        for i in range(0, IMAGE_SIZE, PATCH_SIZE):
            for j in range(0, IMAGE_SIZE, PATCH_SIZE):
                stitched_image[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += denoised_patches[idx]
                count[i:i+PATCH_SIZE, j:j+PATCH_SIZE] += 1
                idx += 1
        denoised_images.append(stitched_image / count)
    return np.array(denoised_images)